# Пайплайн

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
#предупреждение  печати statsmodels (verbose)
warnings.filterwarnings("ignore", category=FutureWarning, module="statsmodels")
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.stattools import grangercausalitytests
from sklearn.ensemble import IsolationForest
from statsforecast.models import (SeasonalNaive, AutoARIMA, ARIMA, AutoTheta, HoltWinters, AutoETS, AutoCES, MSTL, TBATS)
from statsforecast import StatsForecast
from sklearn.preprocessing import StandardScaler
from utilsforecast.losses import mae, rmse, smape, mase
from functools import partial
from utilsforecast.evaluation import evaluate
from sklearn.linear_model import Ridge, LinearRegression, Lasso
#from lightgbm import LGBMRegressor
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM, NHITS, PatchTST
from lightgbm import LGBMRegressor

c:\Users\Acer\Desktop\github\melekhin-time-series\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-09 10:10:57,515	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-06-09 10:10:57,843	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


# Обработка данных

In [2]:
df = pd.read_csv("climate_energy.csv")

#Проверка значение
print(f'Монтонность: {df.index.is_monotonic_increasing}')
print(f'Дубликаты: {df.index.has_duplicates}')
print(f'Пропуски: {df.isna().sum().sum()}')

df['date'] = pd.to_datetime(df['date'], format='mixed', dayfirst=True)
df.set_index('date', inplace=True)
df = df.asfreq('d')

#Определение сезонности
y = df.resample('W').agg({
    'energy_consumption': 'mean',
})
y = y.iloc[1:-2]
SEASON = 52
print(f"Сезонность для энергопотребления : {SEASON}")

Монтонность: True
Дубликаты: False
Пропуски: 0
Сезонность для энергопотребления : 52


In [3]:
#Проверка значимости для экзогенных факторов
# создадим словарь для хранения компонентов
decomposed = {}
df_ = df.copy()
system_cols = ['energy_consumption']
ex_cols = ['co2_emission', 'avg_temperature','humidity', 'urban_population', 'energy_price', 'industrial_activity_index', 'renewable_share']

# функция для многосезонного декомпозиции
def multi_seasonal_decompose(series, weekly=7, yearly=365):
    result = pd.DataFrame(index=series.index)

    # годовая сезонность
    yearly_comp = seasonal_decompose(series, period=yearly, model='additive', extrapolate_trend='freq')
    result['trend_yearly'] = yearly_comp.trend
    result['season_yearly'] = yearly_comp.seasonal
    result['resid_yearly'] = yearly_comp.resid

    return result

# применяем к системным колонкам
for col in system_cols+ex_cols:
    decomposed[col] = multi_seasonal_decompose(df_[col])
    df_[col + '_trend'] = decomposed[col]['trend_yearly']
    df_[col + '_season'] = decomposed[col]['season_yearly']
    df_[col + '_resid'] = decomposed[col]['resid_yearly']  # остатки после всех сезонностей
    df_[col + '_detrend'] = df_[col + '_season']+df_[col + '_resid']

df_resid = df_[
        [f'{c}_resid' for c in system_cols] +
        [f'{c}_resid' for c in ex_cols]].dropna()


def granger_min_pvalue(df, cause, effect, max_lag=24):
    """
    Возвращает минимальный p-value по всем лагам и позицию лага
    """
    data = df[[effect, cause]].dropna()
    results = grangercausalitytests(data, maxlag=max_lag, verbose=False)
    pvals = [results[l][0]['ssr_ftest'][1] for l in results]
    return np.min(pvals), np.argmin(pvals)

# Списки причин и следствий
causes = [f'{c}_resid' for c in ex_cols]
effects = [f'{c}_resid' for c in system_cols]

# Собираем результаты
results = []

for cause in causes:
    for effect in effects:
        res = granger_min_pvalue(df_resid, cause=cause, effect=effect, max_lag=24)
        pval, lag = res
        results.append({
            'cause': cause,
            'effect': effect,
            'pval': pval,
            'lag': lag
        })

# Создаём DataFrame
granger_df = pd.DataFrame(results)

# Вытаскиваем p-значения (игнорируя NaN)
pvals = granger_df['pval'].dropna()
original_idx = granger_df.dropna().index

# Применяем FDR-коррекцию (Benjamini-Hochberg)
_, pvals_corrected, _, _ = multipletests(pvals.values, alpha=0.05, method='fdr_bh')

# Добавляем откорректированные p-значения в DataFrame
granger_df.loc[original_idx, 'pval_corrected'] = pvals_corrected

# Фильтруем значимые результаты (после коррекции)
significant = granger_df[granger_df['pval_corrected'] < 0.05]

# Выводим результаты
if not significant.empty:
    print("Значимые Granger-причинности после FDR-коррекции (p < 0.05):")
    for _, row in significant.iterrows():
        print(f"{row['cause']} → {row['effect']} | p_corrected: {row['pval_corrected']:.3e}, lag: {row['lag']}")
else:
    print("Нет значимых Granger-причинностей после FDR-коррекции.")

Значимые Granger-причинности после FDR-коррекции (p < 0.05):
avg_temperature_resid → energy_consumption_resid | p_corrected: 4.773e-02, lag: 4
humidity_resid → energy_consumption_resid | p_corrected: 2.460e-02, lag: 22
urban_population_resid → energy_consumption_resid | p_corrected: 2.460e-02, lag: 17
energy_price_resid → energy_consumption_resid | p_corrected: 2.460e-02, lag: 12


In [4]:
#Удалим неинформативные столбцы и сохраним обработанную таблицу
df = df.drop(columns = ['industrial_activity_index', 'renewable_share', 'co2_emission'])
df.to_csv('climate_EDA.csv', index_label="date")

# Анализ аномалий

In [5]:
df = pd.read_csv('climate_EDA.csv', parse_dates=["date"], index_col = "date")
ts = ['energy_consumption']
exog_cols = ['avg_temperature', 'humidity','urban_population', 'energy_price']
base_cols = ['unique_id', 'ds', 'y']

y = df.resample('W').agg({
    'energy_consumption': 'mean',
    'avg_temperature': 'mean',
    'urban_population': 'mean',
    'humidity': 'mean',
    'energy_price': 'sum',
})

y = y.iloc[1:-2]

date_index = y.index

panel_list = [
    pd.DataFrame({
        'ds': y.index,
        'y': y[t].values,
        'unique_id': t,
        # Добавляем внешние переменные из y:
        'avg_temperature': y['avg_temperature'].values,
        'urban_population': y['urban_population'].values,
        'humidity': y['humidity'].values,
        'energy_price': y['energy_price'].values
    })
    for t in ts
]

panel_df = pd.concat(panel_list, ignore_index=True)
panel_df = panel_df[base_cols + exog_cols]

FH = int(0.42 * y.shape[0])
df_test = panel_df[base_cols].groupby("unique_id").tail(FH)
df_train = panel_df[base_cols].drop(df_test.index).reset_index(drop=True)

FREQ = y.index.freq
SEASON = 52
horizon = 52

In [6]:
levels = [99]
models = [MSTL(season_length=[SEASON], alias='MSTL')]
alias = [x.alias for x in models]

sf = StatsForecast(
    models=models,
    freq=FREQ,
)

fcst = sf.forecast(df=df_train, h=FH, level=levels, fitted=True)

insample_forecasts = sf.forecast_fitted_values().drop(columns=['y'])

res_df = panel_df.merge(pd.concat([insample_forecasts,fcst],), on=['unique_id', 'ds'], how='left')
res_df = res_df.drop(columns=['humidity', 'urban_population',	'energy_price', 'avg_temperature'],)
res_df['residual'] = res_df['y'] - res_df['MSTL']
anomalies_mstl = res_df[~res_df['y'].between(res_df['MSTL-lo-99'], res_df['MSTL-hi-99'])][['unique_id', 'ds', 'y']].copy()
anomalies_mstl['MSTL_anomaly'] = 1

In [7]:
def detect_isolation_forest_anomlies(df):
  for lag in [7, 26, 52]:
    df[f'lag_{lag}'] = df['y'].shift(lag)
    df[f'lag_{lag}'] = df['y'].shift(lag)
  scaler = StandardScaler()
  df_scaled = scaler.fit_transform(df)
  iso_forest = IsolationForest(n_estimators = 100, contamination=0.01, random_state=42)
  iso_predict = iso_forest.fit_predict(df_scaled)
  anomalies = df.index[iso_predict == -1]
  return anomalies

df_iso_en = panel_df[['ds', 'y']].copy().set_index('ds')

anomalies_if = detect_isolation_forest_anomlies(df_iso_en)

In [8]:
def detect_iqr_anomalies(df):
  Q1 = df.quantile(0.25)
  Q3 = df.quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  df_iqr = (df < lower_bound) | (df > upper_bound)
  return df_iqr

df_en = panel_df[['ds', 'y']].copy().set_index('ds')
result_iqr = detect_iqr_anomalies(df_en)
anomalies_iqr = result_iqr[result_iqr['y']==True]

In [9]:
dates_mstl = [d.strftime('%Y-%m-%d') for d in list(anomalies_mstl['ds'])]
dates_iqr = [d.strftime('%Y-%m-%d') for d in list(anomalies_iqr.index)]
print(f'Даты аномалий вероятностного прогнозирования: {dates_mstl}')
print(f'Даты аномалий IQR: {dates_iqr}')
print(f'Даты аномалий IsolationForest: {list(anomalies_if.strftime("%Y-%m-%d"))}')

Даты аномалий вероятностного прогнозирования: ['2021-03-14', '2023-03-12', '2023-03-19', '2023-04-02', '2023-04-23', '2023-08-06', '2023-08-13', '2023-10-01', '2024-01-07', '2024-02-11', '2024-03-03', '2024-03-24', '2024-04-21', '2024-05-05']
Даты аномалий IQR: []
Даты аномалий IsolationForest: ['2023-03-26', '2024-03-31', '2024-06-23']


# Статистические методы

In [10]:
metrics = [mae, rmse, smape]
season_mase = partial(mase, seasonality=SEASON)
metrics +=[season_mase]

In [11]:
models=[
        ARIMA (order=(4,0,3), season_length=SEASON, seasonal_order=(1, 0, 0)),
        AutoARIMA(season_length=SEASON),
        SeasonalNaive(season_length=SEASON),
        #AutoTheta(season_length=SEASON, decomposition_type="multiplicative"),
        HoltWinters(season_length=SEASON, error_type="A", alias="HW_Add"),
        AutoETS(season_length=SEASON, model='ZZA', damped = False, alias='ETS'),
        AutoCES(season_length=SEASON),
        MSTL(season_length=SEASON,  ),
        TBATS(season_length = SEASON, use_damped_trend = True),
  ]

sf_base = StatsForecast(
    models=models,
    freq=FREQ,
)
LEVELS = [75, 90, 95]

fcst_sf = sf_base.forecast(df=df_train, h=horizon, level=LEVELS)
eval_sf = df_test.merge(fcst_sf, on=['unique_id', 'ds'])

metrics_base = evaluate(
    df=eval_sf,
    train_df=df_train,
    metrics=metrics,
).set_index('metric')

metrics_base

,unique_id,ARIMA,AutoARIMA,SeasonalNaive,HW_Add,ETS,CES,MSTL,TBATS
metric,,,,,,,,,
mae,energy_consumption,1530.013309,1514.530852,1514.530852,1428.863857,1372.840673,1264.198233,1234.346288,1304.561315
rmse,energy_consumption,1889.227688,1989.303224,1989.303224,1788.843453,1723.999926,1668.261275,1602.454590,1600.685763
smape,energy_consumption,0.102077,0.101562,0.101562,0.096961,0.090285,0.081745,0.080068,0.087153
mase,energy_consumption,1.046079,1.035493,1.035493,0.976922,0.938619,0.864339,0.843929,0.891936


In [12]:
date_features = ['week']
models={
        'lasso': Lasso(),
        'lin_reg': LinearRegression(),
        'ridge': Ridge(alpha=0.10),
        'lgbm':LGBMRegressor(n_estimators=100, verbosity=-1),
}

LEVELS = [75, 90, 95]

fcst = MLForecast(
    models = models,
    freq=FREQ,
    date_features = date_features,
    lags=[26],
    target_transforms=[Differences([26])],
)
fcst.fit(df_train, )
fcst_sf = fcst.predict(h=horizon, )

eval_mf = df_test[['unique_id', 'ds', 'y']].merge(
    fcst_sf,
    on=['unique_id', 'ds'],
    how='inner'
)

metrics = [mae, rmse, smape]
season_mase = partial(mase, seasonality=SEASON)
metrics +=[season_mase]
ml_metrics = evaluate(
    df=eval_mf,
    metrics=metrics,
    train_df=df_train
).set_index('metric')

ml_metrics

c:\Users\Acer\Desktop\github\melekhin-time-series\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Не удается найти указанный файл
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Acer\Desktop\github\melekhin-time-series\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Acer\Desktop\github\melekhin-time-series\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Acer\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548,

,unique_id,lasso,lin_reg,ridge,lgbm
metric,,,,,
mae,energy_consumption,1361.488034,1361.501649,1361.501477,1374.506873
rmse,energy_consumption,1740.672556,1740.690792,1740.690561,1692.365775
smape,energy_consumption,0.091047,0.091048,0.091048,0.093267
mase,energy_consumption,0.930857,0.930866,0.930866,0.939758


In [13]:
import torch
DEVICE = "gpu" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
DEVICE

'cpu'

In [14]:
test_size = SEASON
#horizon = SEASON
val_size  = test_size//2
log_path = ''

models = [
        PatchTST(h=horizon, input_size = 26, max_steps=50, learning_rate = 0.001, patch_len=7, n_heads = 16, random_seed=42),
]

nf = NeuralForecast(models=models, freq=FREQ)
nf.fit(df=df_train, val_size=0)
y_hat = nf.predict(df_train)

eval_dl = df_test[['unique_id', 'ds', 'y']].merge(
    y_hat,
    on=['unique_id', 'ds'],
    how='inner'
)

dl_metrics = evaluate(
    df=eval_dl,
    metrics=metrics,
    train_df=df_train
).set_index('metric')
dl_metrics

Seed set to 42
c:\Users\Acer\Desktop\github\melekhin-time-series\.venv\Lib\site-packages\neuralforecast\common\_base_model.py:602: UserWarning: val_check_steps is greater than max_steps, setting val_check_steps to max_steps.
  warnings.warn(
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name         | Type              | Params | Mode 
-----------------------------------------------------------
0 | loss         | MAE               | 0      | train
1 | padder_train | ConstantPad1d     | 0      | train
2 | scaler       | TemporalNorm      | 0      | train
3 | model        | PatchTST_backbone | 425 K  | train
-----------------------------------------------------------
425 K     Trainable params
3         Non-trainable params
425 K     Total params
1.703     Total estimated model params size (MB)
90        Modules in train mode
0         Modules in eval mode


c:\Users\Acer\Desktop\github\melekhin-time-series\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 49: 100%|██████████| 1/1 [00:00<00:00,  3.31it/s, v_num=187, train_loss_step=907.0, train_loss_epoch=907.0]    

`Trainer.fit` stopped: `max_steps=50` reached.


Epoch 49: 100%|██████████| 1/1 [00:00<00:00,  3.28it/s, v_num=187, train_loss_step=907.0, train_loss_epoch=907.0]

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
c:\Users\Acer\Desktop\github\melekhin-time-series\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 111.10it/s]


,unique_id,PatchTST
metric,,
mae,energy_consumption,1419.723577
rmse,energy_consumption,1712.246810
smape,energy_consumption,0.093431
mase,energy_consumption,0.970673


In [15]:
test_size = SEASON
val_size  = test_size//2
log_path = ''

models = [
    LSTM(h=horizon, input_size = 26, max_steps=50, encoder_n_layers=2, learning_rate=0.001,  scaler_type='standard', random_seed=42),
    NHITS(h=horizon, input_size = 26, max_steps=50, n_freq_downsample=[4,4,4], learning_rate=0.001, random_seed=42),
    PatchTST(h=horizon, input_size = 26, max_steps=50, learning_rate = 0.001, patch_len=7, n_heads = 16, random_seed=42),
]

nf = NeuralForecast(models=models, freq=FREQ)
nf.fit(df=df_train, val_size=0)
y_hat = nf.predict(df_train)

eval_dl = df_test[['unique_id', 'ds', 'y']].merge(
    y_hat,
    on=['unique_id', 'ds'],
    how='inner'
)

dl_metrics = evaluate(
    df=eval_dl,
    metrics=metrics,
    train_df=df_train
).set_index('metric')
dl_metrics

Seed set to 42
Seed set to 42
Seed set to 42
c:\Users\Acer\Desktop\github\melekhin-time-series\.venv\Lib\site-packages\neuralforecast\common\_base_model.py:602: UserWarning: val_check_steps is greater than max_steps, setting val_check_steps to max_steps.
  warnings.warn(
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name              | Type          | Params | Mode 
------------------------------------------------------------
0 | loss              | MAE           | 0      | train
1 | padder_train      | ConstantPad1d | 0      | train
2 | scaler            | TemporalNorm  | 0      | train
3 | hist_encoder      | LSTM          | 199 K  | train
4 | mlp_decoder       | MLP           | 16.6 K | train
5 | upsample_sequence | Linear        | 1.4 K  | train
------------------------------------------------------------
217 K     Trainable params
0         Non-trainable params
217 K     Total params
0.869     Total estimated model params size (MB)
11        Modul

c:\Users\Acer\Desktop\github\melekhin-time-series\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 49: 100%|██████████| 1/1 [00:00<00:00, 16.13it/s, v_num=189, train_loss_step=0.720, train_loss_epoch=0.720]

`Trainer.fit` stopped: `max_steps=50` reached.


Epoch 49: 100%|██████████| 1/1 [00:00<00:00, 15.38it/s, v_num=189, train_loss_step=0.720, train_loss_epoch=0.720]


...

GPU available: False, used: False
TPU available: False, using: 0 TPU cores



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 199.94it/s]


GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 199.81it/s]


,unique_id,LSTM,NHITS,PatchTST
metric,,,,
mae,energy_consumption,1152.628533,1156.044255,1419.723577
rmse,energy_consumption,1458.999373,1424.361143,1712.246810
smape,energy_consumption,0.076369,0.077249,0.093431
mase,energy_consumption,0.788059,0.790394,0.970673


In [16]:
avg_by_series = metrics_base.groupby(['unique_id', 'metric']).mean(numeric_only=True)
avg_by_series.style.background_gradient(cmap='RdYlGn_r', axis=1).format(precision=2)

In [17]:
avg_by_series_ml = ml_metrics.groupby(['unique_id', 'metric']).mean(numeric_only=True)
avg_by_series_ml.style.background_gradient(cmap='RdYlGn_r', axis=1).format(precision=2)

In [18]:
avg_by_series_dl = dl_metrics.groupby(['unique_id', 'metric']).mean(numeric_only=True)
avg_by_series_dl.style.background_gradient(cmap='RdYlGn_r', axis=1).format(precision=2)